#### Gold Publish

Purpose

- Read Silver Delta
- Publish trusted data
- Exclude Quarantine
- Create Gold Data Product

Output

/Volumes/assignment/university/gold/university_chapters/v1

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime
import uuid

In [0]:
silver_path = "/Volumes/assignment/university/silver/university_chapters"

gold_path = "/Volumes/assignment/university/gold/university_chapters/v1"

In [0]:
silver_df = (
    spark.read
         .format("delta")
         .load(silver_path)
)

display(silver_df)

source_object_id,chapter_id,chapter_name,city,state,longitude,latitude,longitude/latitude,dq_status,dq_warnings,processed_timestamp,source_run_folder
19,CA-0355,California Polytechnic State University,San Luis Obispo,CA,-120.66319100299995,35.274309145000075,"-120.663191,35.274309",OK,List(),2026-07-22T05:15:56.844Z,dbfs:/Volumes/assignment/university/bronze/university_chapters/20260722_031302_11b5b2b6/
26,CA-0300,Chico State University,Chico,CA,-121.83546324499997,39.73998176200007,"-121.835463,39.739982",OK,List(),2026-07-22T05:15:56.844Z,dbfs:/Volumes/assignment/university/bronze/university_chapters/20260722_031302_11b5b2b6/
1757,CA-0362,Fresno State,Fresno,CA,-119.73959481924653,36.82354541564933,"-119.739595,36.823545",OK,List(),2026-07-22T05:15:56.844Z,dbfs:/Volumes/assignment/university/bronze/university_chapters/20260722_031302_11b5b2b6/


In [0]:
gold_df = silver_df.select(
    "source_object_id",
    "chapter_id",
    "chapter_name",
    "city",
    "state",
    "longitude/latitude"
    )

In [0]:
display(gold_df)

source_object_id,chapter_id,chapter_name,city,state,longitude/latitude
19,CA-0355,California Polytechnic State University,San Luis Obispo,CA,"-120.663191,35.274309"
26,CA-0300,Chico State University,Chico,CA,"-121.835463,39.739982"
1757,CA-0362,Fresno State,Fresno,CA,"-119.739595,36.823545"


In [0]:
(
    gold_df.write
        .format("delta")
        .mode("overwrite")
        #.option("overwriteSchema", "true")
        .save(gold_path)
)

In [0]:
display(
    spark.read
         .format("delta")
         .load(gold_path)
)

source_object_id,chapter_id,chapter_name,city,state,longitude/latitude
19,CA-0355,California Polytechnic State University,San Luis Obispo,CA,"-120.663191,35.274309"
26,CA-0300,Chico State University,Chico,CA,"-121.835463,39.739982"
1757,CA-0362,Fresno State,Fresno,CA,"-119.739595,36.823545"


In [0]:
gold_rows = gold_df.count()

ok_rows = gold_df.filter(
    col("dq_status")=="CLEAN"
).count()

warning_rows = gold_df.filter(
    col("dq_status")=="WARNING"
).count()

print(f"Gold Rows        : {gold_rows}")
print(f"Clean Rows       : {ok_rows}")
print(f"Warning Rows     : {warning_rows}")

Gold Rows        : 3
Clean Rows       : 0
Warning Rows     : 0


In [0]:
display(gold_df.count())

3

In [0]:
display(
    gold_df.filter(
        (~col("longitude").between(-180,180)) | (~col("latitude").between(-90,90))
    )
)

source_object_id,chapter_id,chapter_name,city,state,longitude/latitude


In [0]:
display(

    gold_df.filter(
        col("dq_status")=="WARNING"
    )

)

source_object_id,chapter_id,chapter_name,city,state,longitude/latitude


In [0]:
gold_df.groupBy("state").count().show()

+-----+-----+
|state|count|
+-----+-----+
|   CA|    3|
+-----+-----+

